# 8장 코드 읽기 훈련 — 남의 코드, AI의 코드

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch08_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | 8.2의 1단계를 직접 수행(정의 목록·호출 순서를 손으로 적기), 8.3의 `--dry-run` 실행 | [ch08_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch08_1_follow.ipynb) |
| 2. 주석 달기 | `mnist_main.py` 의 `main()` 에 "준비물" 단위로 주석 달기: 인자 / 장치 / 데이터 / 모델 / 옵티마이저 / 반복 / 저장 | [ch08_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch08_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 8.5의 AI 설명 대조를 수행해 틀린 곳 목록을 파일로 남기고, 8.4의 조각을 다시 쓴 파일과 함께 저장소에 커밋 | 책을 보며 진행 (노트북 없음) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "08"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/08")

`main()` 에 「준비물」 단위로 주석을 다세요: 인자 / 장치 / 데이터 / 모델 / 옵티마이저 / 반복 / 저장.

## `mnist_main.py` — 대상: `main`

In [ ]:
%%writefile mnist_main.py
# 8장 코드 읽기 훈련의 대상 파일.
# 출처: pytorch/examples 저장소의 mnist/main.py (https://github.com/pytorch/examples/blob/main/mnist/main.py)
#       2026-09-06 main 브랜치에서 가져옴. Copyright (c) 2017, Pytorch contributors. All rights reserved. BSD 3-Clause 라이선스로 배포되며, 조건과 면책 조항은 같은 폴더의 LICENSE-pytorch-examples.txt 에 있다(이 파일을 옮길 때 함께 옮긴다).
#       이 파일은 학습용 읽기 자료로 그대로 두었고, 주석만 이 책의 8장 실습을 위해 독자가 덧붙인다.
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if args.dry_run:
                break


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))


def main():
    parser = argparse.ArgumentParser(description='PyTorch MNIST Example')
    parser.add_argument('--batch-size', type=int, default=64, metavar='N',
                        help='input batch size for training (default: 64)')
    parser.add_argument('--test-batch-size', type=int, default=1000, metavar='N',
                        help='input batch size for testing (default: 1000)')
    parser.add_argument('--epochs', type=int, default=14, metavar='N',
                        help='number of epochs to train (default: 14)')
    parser.add_argument('--lr', type=float, default=1.0, metavar='LR',
                        help='learning rate (default: 1.0)')
    parser.add_argument('--gamma', type=float, default=0.7, metavar='M',
                        help='Learning rate step gamma (default: 0.7)')
    parser.add_argument('--no-accel', action='store_true',
                        help='disables accelerator')
    parser.add_argument('--dry-run', action='store_true',
                        help='quickly check a single pass')
    parser.add_argument('--seed', type=int, default=1, metavar='S',
                        help='random seed (default: 1)')
    parser.add_argument('--log-interval', type=int, default=10, metavar='N',
                        help='how many batches to wait before logging training status')
    parser.add_argument('--save-model', action='store_true',
                        help='For Saving the current Model')
    args = parser.parse_args()

    use_accel = not args.no_accel and torch.accelerator.is_available()
    torch.manual_seed(args.seed)

    if use_accel:
        device = torch.accelerator.current_accelerator()
    else:
        device = torch.device("cpu")

    train_kwargs = {'batch_size': args.batch_size}
    test_kwargs = {'batch_size': args.test_batch_size}
    if use_accel:
        accel_kwargs = {'num_workers': 1,
                        'persistent_workers': True,
                       'pin_memory': True,
                       'shuffle': True}
        train_kwargs.update(accel_kwargs)
        test_kwargs.update(accel_kwargs)

    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
        ])
    dataset1 = datasets.MNIST('../data', train=True, download=True,
                       transform=transform)
    dataset2 = datasets.MNIST('../data', train=False,
                       transform=transform)
    train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
    test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=args.lr)
    scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)

    for epoch in range(1, args.epochs + 1):
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    if args.save_model:
        torch.save(model.state_dict(), "mnist_cnn.pt")


if __name__ == '__main__':
    main()

### 확인

In [ ]:
!python mnist_main.py --dry-run --no-accel --epochs 1

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- mnist_main.py